# Feature Engineering with information available at renewal

## Overview

- The goal of this notebook is to engineer features from variables of dataset that have been obtained by an underwriter(s) at least first renewal. This implies that we may now have more internal data available our our disposal.
- This will be tackled in 4 waves in terms of the data now available to us
  - Driver
  - Car Data
  - Payment history
  - Claims history

## Setup

In [ ]:
import os
from datetime import datetime
from typing import Any
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
os.chdir('..') ## on the assumption that this is rum from notebook/freq_sev

### Load the data

In [ ]:
insurance_initiation_variables_path = "data/input/Motor_vehicle_insurance_data.csv"
claims_variables_path = "data/input/exp/sample_type_claim.csv"
insurance_df = pd.read_csv(insurance_initiation_variables_path, delimiter=';')
claims_df = pd.read_csv(claims_variables_path, delimiter=';')

## 1. Prepare Dataset

| Step | Action |
|------|--------|
| Aggregate | Sum claims by (ID, year) |
| Merge | Left join on (ID, year) |
| Fill | NaN → 0 for no claims |
| Split | 80/20 train-test split |

### 1.1 Aggregate claims by policyholder and year

In [ ]:
claim_grouping_columns = ['ID', 'Cost_claims_year']
claim_aggregation_column = 'Cost_claims_by_type'
claims_aggregated = (
    claims_df
    .groupby(claim_grouping_columns, as_index=False)[claim_aggregation_column]
    .sum()
)

### 1.2 Merge insurance and claims data

In [ ]:
merging_columns = ['ID', 'Cost_claims_year']

dataset = insurance_df.merge(claims_aggregated, on=merging_columns, how='left')
dataset[claim_aggregation_column] = dataset[claim_aggregation_column].fillna(0)
dataset['claims_frequency'] = (dataset[claim_aggregation_column] > 0).astype(int)

### 1.3 Split into train and test sets

In [ ]:
test_ratio = 0.2
to_shuffle = False
if to_shuffle:
    dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

split_index = int(len(dataset) * (1 - test_ratio))
trainset = dataset.iloc[:split_index].reset_index(drop=True)
testset = dataset.iloc[split_index:].reset_index(drop=True)
print(f"Total records: {len(dataset)}")
print(f"Training set: {len(trainset)} records ({100*(1-test_ratio):.0f}%)")
print(f"Test set: {len(testset)} records ({100*test_ratio:.0f}%)")

## 2. Feature Engineering in Waves

While there are now advancement in the motor insurance sector that enables using telematic data points, I will be exploring traditional features in this segment mostly based on domain knowledge [some capture here](https://www.researchgate.net/publication/338007809_An_Analysis_of_the_Risk_Factors_Determining_Motor_Insurance_Premium_in_a_Small_Island_State_The_Case_of_Malta).

### 2.1 - Driver Features
The below are directly available in the dataset
- Driver DOB
- Driver Driving License date
- Second driver on the car

Some features to engineer from the above
- Respective driver age at inception of each contract
- Evidence backed driver age banding at the inception of each contract
- Length of driving license (driving experience) at the inception of each contract
- Evidence backed license length banding at the inception of each contract.
- Interaction between second driver & driver age banding - as a proxyto identify couples or parent/children?
    - e.g young single driver may be considered high risk
    - older driver with second driver may be low to moderate risk (although we cant conclude with this)
- Age experience gap for each contract. Difference between age and driving experience in years. Lower is better? But how low?
    - So a 25 year old with 1 year experience has agegapexp = 24
    - 25 years old with 6 years experience has agegapexp=19

## 3. Implement Engineered Features

| Feature | Formula | Unit |
|---------|---------|------|
| Driver_age_years | today - Date_birth | Years |
| Driver_experience_years | today - Date_driving_licence | Years |
| Car_age_years | today_year - Year_matriculation | Years |
| power_to_weight | Power / Weight | HP/kg |

### 3.1 Define helper functions

In [ ]:
# def convert_to_datetime(value:object, format:str="%d/%m/%Y", yearfirst:bool=True) -> Any:
#     return pd.to_datetime(arg=value, format=format, yearfirst=yearfirst)

# def take_datetime_difference_in_years(first_datetime:datetime, second_datetime:datetime, interval) -> float:
#     diff = (second_datetime - first_datetime) / np.timedelta64(1, interval)
#     diff_years = diff/365.25
#     return diff_years

# def take_int_difference(first_number:int, second_number:int) -> int:
#     return abs(first_number - second_number)

# def calculate_vehicle_age(vehicle_matriculation:int):
#     pass

### 3.2 Apply transformations to trainset

In [ ]:
# today_date = pd.Timestamp.today()  ## why not consider the end date highlighted for the data collection?n- 2018-12-31
# today_year = today_date.year
# features_trainset = (
#     trainset
#     .assign(
#         Date_birth_dt=trainset['Date_birth'].apply(convert_to_datetime),
#         Date_driving_licence_dt=trainset['Date_driving_licence'].apply(convert_to_datetime),
#         power_to_weight = trainset['Power'] / trainset['Weight'],
#         Car_age_years= trainset['Year_matriculation'].apply(take_int_difference, args=(today_year,)),
#         Type_fuel_encoded=trainset['Type_fuel'].map({'P': 0, 'D': 1}))
#     .assign(
#         Driver_age_years=lambda df: df['Date_birth_dt'].apply(take_datetime_difference_in_years, args=(today_date, 'D')),
#         Driver_experience_years=lambda df: df['Date_driving_licence_dt'].apply(take_datetime_difference_in_years, args=(today_date, 'D')),
#     )
# )

### 3.3 Resulting variables

At the end of engineering on the features on the trainset, we now have the following variables in our dataset
- ID
- Date_birth
- Date_driving_licence
- Distribution_channel
- Premium
- Cost_claims_year
- Type_risk
- Area
- Second_driver
- Year_matriculation
- Power
- Cylinder_capacity
- Value_vehicle
- N_doors
- Type_fuel
- Length
- Weight
- claims_frequency
- Date_birth_dt
- Date_driving_licence_dt
- power_to_weight
- Car_age_years
- Driver_age_years
- Driver_experience_years
- Type_fuel_encoded (P=0, D=1)

For all intent and purposes, the premium, cost_claims_year and claims frequency are potential target variables

Some more features to think about
- Vehicle age
- Chosen payment frequency
- Fuel type encoding
- Vehicle type - encoding
- no claims discount - good indicator of future claims - this information is often collected at initial point but do we have it in this dataset

In [ ]:
# ### Go back to read the full dataset.
# ## Get the date of policy start to calculage the age of the car at the point of first contact
# ## Bring in the payment frequency
# ## Encoding of vehicle type and fuel type

## 4. Explore Engineered Features

| Check | Purpose |
|-------|---------|
| Null counts | Feature completeness |
| Histograms | Distribution shape |
| Bar plots | Categorical balance |
| Correlation | Multicollinearity |

### 4.1 Check feature completeness

In [ ]:
# #Feature completeness pattern
# features_trainset.isnull().sum()

### 4.2 Binning pattern for histogram

In [ ]:
# ##Feature data distribution pattern - binning pattern for histogram
# #Step 1: Bin the variable
# variable = 'Weight'
# binned_variable = pd.cut(features_trainset[variable], bins=10)

# #Step 2: join the binned variable to existing dataset
# binned_variable.name = f"binned_{variable}"
# binned_df = pd.concat([features_trainset, binned_variable], axis=1 )
# #binned_df

# #Step 3: Group the dataset using the binned variable
# groups = []
# for group, subset in features_trainset.groupby(by=binned_variable, observed=False):
#     groups.append({
#         'Binrange': group,
#         'Count': len(subset),
#     })
# group_df = pd.DataFrame(groups)

# #Step 4: Visualize this with histogram using barplot (converted the bin range to str because matplotlib not handling intervals well)
# plt.bar(x=group_df['Binrange'].astype(str), height=group_df['Count'])
# plt.xticks(rotation=90)
# plt.show()

### 4.3 Histogram visualisation (scalable)

In [ ]:
# ## Easier patter for histogram visualization with seaborn that scales to columns
# #Step 1: Define the columns to obtain distribution as list
# cols = ['Car_age_years', 'Driver_age_years', 'Driver_experience_years', 'Power', 'power_to_weight']

# ##Step 2: Define the bin (in this case number of bins)
# bin = 10

# #Step 3 : visualize
# fig, axes = plt.subplots(1, len(cols), figsize=(18, 3))
# for i, col in enumerate(cols):
#     sns.histplot(data=features_trainset, x=col, bins=bin, kde=False, ax=axes[i])
#     axes[i].set_title(f"Distribution of {col}")
# plt.tight_layout()
# plt.show()

### 4.4 Categorical distributions

In [ ]:
# ##Similar pattern this time for barplots
# cols  = [
#     'Distribution_channel',
#     'Type_risk',
#     'Type_fuel',
#     'Area',
#     'Second_driver'
# ]

# fig, axes = plt.subplots(2, 3, figsize=(15, 6))  # 2 rows, 3 columns
# axes = axes.flatten()  # turn into 1D array

# for i, col in enumerate(cols):
#     sns.countplot(data=features_trainset, x=col, ax=axes[i])
#     axes[i].set_title(f"Distribution of {col}")
#     axes[i].tick_params(axis="x")
# fig.delaxes(axes[-1])

# plt.tight_layout()
# plt.show()

### 4.5 Bivariate correlation

In [ ]:
# ## Simple bivariate relationship pattern
# features_trainset[[
#                    'claims_frequency','Power', 'Cylinder_capacity','power_to_weight',
#                    'Value_vehicle', 'Length', 'Weight', 'Type_fuel_encoded',
#                     'Driver_age_years','Payment','Driver_experience_years',
#                     'Type_risk', 'Second_driver', 'Car_age_years', 'Area',
#                    'Distribution_channel', 'N_doors']].corr(method="spearman")

### 4.6 Why Weak Correlations Don't Mean Useless Features

A Spearman correlation of 0.02 means almost nothing linearly, but **claims frequency patterns are rarely linear**. Let's explore the actual claims rate by experience bands:

In [ ]:
# # Check claims rate (base rate) first
# base_claims_rate = features_trainset['claims_frequency'].mean()
# print(f"Base claims rate: {base_claims_rate:.2%}")
# print(f"This means {100-base_claims_rate*100:.1f}% of records have NO claims - correlation will be weak!")

In [ ]:
# # Explore claims rate by Driver Experience bands
# exp_bins = [0, 2, 5, 10, 20, 30, 100]
# exp_labels = ['0-2yr', '3-5yr', '6-10yr', '11-20yr', '21-30yr', '30+yr']

# features_trainset['Experience_band'] = pd.cut(
#     features_trainset['Driver_experience_years'],
#     bins=exp_bins,
#     labels=exp_labels
# )

# # Calculate claims rate per band
# claims_by_exp = features_trainset.groupby('Experience_band', observed=False).agg(
#     claims_rate=('claims_frequency', 'mean'),
#     count=('claims_frequency', 'count'),
#     total_claims=('claims_frequency', 'sum')
# ).round(4)

# print("Claims Rate by Driver Experience:")
# print(claims_by_exp)
# print()

# # Visualize - this shows patterns correlation misses!
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# # Left: Claims rate by experience
# claims_by_exp['claims_rate'].plot(kind='bar', ax=axes[0], color='steelblue')
# axes[0].axhline(y=base_claims_rate, color='red', linestyle='--', label=f'Base rate: {base_claims_rate:.2%}')
# axes[0].set_ylabel('Claims Rate')
# axes[0].set_xlabel('Driver Experience')
# axes[0].set_title('Claims Rate by Experience Band\n(Red line = overall average)')
# axes[0].legend()
# axes[0].tick_params(axis='x', rotation=45)

# # Right: Volume by experience (for context)
# claims_by_exp['count'].plot(kind='bar', ax=axes[1], color='coral')
# axes[1].set_ylabel('Number of Policies')
# axes[1].set_xlabel('Driver Experience')
# axes[1].set_title('Policy Volume by Experience Band')
# axes[1].tick_params(axis='x', rotation=45)

# plt.tight_layout()
# plt.show()

### 4.7 The Real Question: Is There a Pattern?

Look at the chart above. Even if correlation is 0.02, ask yourself:
- Do certain experience bands have **higher claims rates than the base rate**?
- Is there a U-shape or declining pattern?

If YES → the feature IS useful, just not linearly correlated.

**Key Insight**: In insurance, we use **GLMs (Poisson regression)** which model the **log of expected claims**, not linear correlation. A feature can have 0.02 Spearman correlation but still be statistically significant in a GLM!
